# LeetCode 507 — Perfect Number bằng Python

**Link đề:** https://leetcode.com/problems/perfect-number/
**Độ khó:** Easy · **Chủ đề:** Math

```python
class Solution:
    def checkPerfectNumber(self, num: int) -> bool:
        ...
```

## Mục tiêu học
1. Hiểu **ước số** (divisor), **ước thực sự** (proper divisor — không tính chính nó), và **số hoàn hảo**.
2. Dùng `%` để kiểm tra chia hết; `math.isqrt` để lấy căn nguyên an toàn.
3. Đi từ duyệt `1..num-1` (O(n), quá chậm với `10^8`) → chỉ tới `num/2` → **ghép cặp ước tới `√num`** (O(√n)).
4. Xử lý cẩn thận các biên: `num = 1`, số chính phương (ước bị đếm hai lần).
5. Biết định lý **Euclid–Euler** → chỉ có 5 số hoàn hảo ≤ `10^8`; so sánh 4 cách, test và bài tập về ước số.

## 1. Đề bài

**Số hoàn hảo** (perfect number) là số nguyên dương **bằng tổng các ước dương của nó, không tính chính nó**.

Cho `num`, trả về `True` nếu `num` là số hoàn hảo, ngược lại `False`.

| num | Output | Giải thích |
|---:|---|---|
| `28` | `True` | `1 + 2 + 4 + 7 + 14 = 28` |
| `7` | `False` | ước thực sự chỉ có `1`, tổng = 1 |

**Constraints:** `1 <= num <= 10^8`.

Ví dụ khác cần nhớ: `6 = 1 + 2 + 3` là số hoàn hảo nhỏ nhất. `1` **không** hoàn hảo (không có ước thực sự nào, tổng = 0).

## 2. Nền tảng

### 2.1. Kiểm tra chia hết bằng `%`

`num % d == 0` ⇔ `d` là ước của `num`.

In [1]:
num = 28
for d in range(1, num + 1):
    if num % d == 0:
        print(d, end=" ")
print()
print("ước thực sự:", [d for d in range(1, num) if num % d == 0])
print("tổng:", sum(d for d in range(1, num) if num % d == 0))

1 2 4 7 14 28 
ước thực sự: [1, 2, 4, 7, 14]
tổng: 28


### 2.2. `sum(...)` với generator

`sum(d for d in ... if ...)` cộng các giá trị thoả điều kiện mà **không tạo list trung gian**.

### 2.3. Ước đi theo **cặp**

Nếu `d` là ước của `num` thì `num // d` cũng là ước. Với `28`:

| d | num // d |
|---:|---:|
| 1 | 28 |
| 2 | 14 |
| 4 | 7 |

Trong mỗi cặp, số nhỏ luôn `≤ √num` (vì nếu cả hai `> √num` thì tích `> num`). Vậy chỉ cần duyệt `d` từ 1 đến `√num` là tìm được **mọi** ước.

In [2]:
num = 28
d = 1
while d * d <= num:
    if num % d == 0:
        print(f"cặp ({d}, {num // d})")
    d += 1

cặp (1, 28)
cặp (2, 14)
cặp (4, 7)


### 2.4. `math.isqrt(n)` — căn bậc hai **nguyên**

`math.isqrt(n)` trả về `⌊√n⌋` chính xác (số nguyên), tránh sai số số thực của `int(n ** 0.5)` với số lớn.

In [3]:
import math
print(math.isqrt(28), math.isqrt(36), math.isqrt(10**8))
print(28 ** 0.5, int(28 ** 0.5))
big = (10**15 + 37) ** 2 - 1      # số rất lớn, sát dưới một số chính phương
print("int(**0.5):", int(big ** 0.5), "| isqrt:", math.isqrt(big), "| đúng là", 10**15 + 36)

5 6 10000
5.291502622129181 5
int(**0.5): 1000000000000037 | isqrt: 1000000000000036 | đúng là 1000000000000036


Thấy không: với số rất lớn, `int(big ** 0.5)` có thể **làm tròn sai lên 1** vì số thực (float) chỉ có ~15–16 chữ số chính xác. Với `num ≤ 10^8` thì vẫn ổn, nhưng nên tập thói quen dùng `isqrt`.

### 2.5. Số chính phương — cẩn thận đếm trùng

`36 = 6·6`: cặp `(6, 6)` chỉ là **một** ước. Khi ghép cặp phải kiểm tra `d != num // d`.

## 3. Cách 1 — Trực giác: thử mọi `d` từ 1 tới `num - 1`

In [4]:
def is_perfect_bruteforce(num):
    total = 0
    for d in range(1, num):
        if num % d == 0:
            total += d
    return total == num

print([n for n in range(1, 10000) if is_perfect_bruteforce(n)])   # các số hoàn hảo < 10^4

[6, 28, 496, 8128]


### Nút thắt

`num` lên tới `10^8` → tối đa `10^8` phép chia cho **một** lần gọi: mất cỡ vài giây trong Python, mà LeetCode chạy hàng chục test → dễ **TLE** (Time Limit Exceeded — quá thời gian).

Cải tiến nhỏ: ước thực sự lớn nhất ≤ `num / 2`, nên duyệt tới `num // 2` — **vẫn O(n)**, chỉ nhanh gấp 2.

Đo nhanh trên số vừa phải để cảm nhận tốc độ:

In [5]:
import time

def is_perfect_half(num):
    return num > 1 and sum(d for d in range(1, num // 2 + 1) if num % d == 0) == num

for f, x in [(is_perfect_bruteforce, 33550336), (is_perfect_half, 33550336)]:
    t0 = time.perf_counter(); r = f(x); t1 = time.perf_counter()
    print(f"{f.__name__:22s}({x}) = {r}  mất {t1 - t0:.2f} s")

is_perfect_bruteforce (33550336) = True  mất 1.17 s


is_perfect_half       (33550336) = True  mất 0.58 s


## 4. Insight — ghép cặp ước tới √num

Duyệt `d = 2..⌊√num⌋`. Mỗi khi `num % d == 0`:
- cộng `d`,
- cộng thêm `num // d` **nếu khác `d`** (tránh đếm trùng với số chính phương).

Ước `1` xử lý riêng (cộng sẵn vào `total = 1`), và **không** cộng `num` (vì là ước của chính nó — cặp `(1, num)`).
Trường hợp `num = 1`: không có ước thực sự → trả `False` ngay (nếu không, `total = 1 == num` sẽ cho `True` **sai**).

Số vòng lặp chỉ còn `√(10^8) = 10^4`.

## 5. Lời giải chính — mổ xẻ từng đoạn

1. `if num <= 1: return False`.
2. `total = 1` (ước 1).
3. `for d in range(2, isqrt(num) + 1)`: nếu chia hết → `total += d`, và nếu `d != num // d` → `total += num // d`.
4. `return total == num`.

In [6]:
def is_perfect_sqrt(num):
    if num <= 1:
        return False
    total = 1
    for d in range(2, math.isqrt(num) + 1):
        if num % d == 0:
            total += d
            other = num // d
            if other != d:
                total += other
    return total == num

print(is_perfect_sqrt(28), is_perfect_sqrt(7), is_perfect_sqrt(1), is_perfect_sqrt(6))
t0 = time.perf_counter(); r = is_perfect_sqrt(33550336); t1 = time.perf_counter()
print(f"is_perfect_sqrt(33550336) = {r}  mất {t1 - t0:.4f} s")

True False False True
is_perfect_sqrt(33550336) = True  mất 0.0002 s


### 5.1. Vì sao điều kiện dừng là `d * d <= num`?

Ba cách viết tương đương về ý nghĩa "d không vượt quá √num":

| Cách viết | Ưu điểm | Lưu ý |
|---|---|---|
| `for d in range(2, math.isqrt(num) + 1)` | rõ ràng, nhanh | nhớ `+ 1` vì `range` không lấy cận trên |
| `while d * d <= num` | không cần import, không có số thực | phải tự `d += 1` |
| `d <= num ** 0.5` | ngắn | dùng số thực, có thể sai với số rất lớn |

Thử với số chính phương `49`: cần xét tới `d = 7`. Nếu quên `+ 1` trong `range`, ta bỏ sót ước `7`.

In [7]:
num = 49
print("range đúng :", list(range(2, math.isqrt(num) + 1)))
print("quên + 1   :", list(range(2, math.isqrt(num))))     # thiếu 7
d, seen = 2, []
while d * d <= num:
    seen.append(d); d += 1
print("while d*d  :", seen)

range đúng : [2, 3, 4, 5, 6, 7]
quên + 1   : [2, 3, 4, 5, 6]
while d*d  : [2, 3, 4, 5, 6, 7]


### 5.2. Đếm số vòng lặp — cảm nhận O(n) so với O(√n)

In [8]:
for num in [100, 10**4, 10**6, 10**8]:
    print(f"num={num:>11,}: duyệt tới num-1 = {num - 1:>11,} vòng | duyệt tới √num = {math.isqrt(num) - 1:>6,} vòng")

num=        100: duyệt tới num-1 =          99 vòng | duyệt tới √num =      9 vòng
num=     10,000: duyệt tới num-1 =       9,999 vòng | duyệt tới √num =     99 vòng
num=  1,000,000: duyệt tới num-1 =     999,999 vòng | duyệt tới √num =    999 vòng
num=100,000,000: duyệt tới num-1 =  99,999,999 vòng | duyệt tới √num =  9,999 vòng


## 6. TRACE từng bước

`num = 28`, `⌊√28⌋ = 5`:

| d | 28 % d | cộng d | other = 28 // d | cộng other? | total sau |
|---:|---:|---|---:|---|---:|
| (khởi tạo) | | | | | 1 |
| 2 | 0 | +2 | 14 | +14 | 17 |
| 3 | 1 | — | — | — | 17 |
| 4 | 0 | +4 | 7 | +7 | 28 |
| 5 | 3 | — | — | — | 28 |

`total = 28 == num` → `True`.

In [9]:
def trace_perfect(num):
    print(f"num = {num}, isqrt = {math.isqrt(num) if num > 0 else 0}")
    if num <= 1:
        print("num <= 1 -> False")
        return False
    total = 1
    print("khởi tạo total = 1 (ước 1)")
    for d in range(2, math.isqrt(num) + 1):
        if num % d == 0:
            other = num // d
            add = d + (other if other != d else 0)
            total += add
            note = f"+{d}" + (f" +{other}" if other != d else " (số chính phương, không cộng trùng)")
            print(f"  d={d}: chia hết, {note:40s} total={total}")
        else:
            print(f"  d={d}: không chia hết (dư {num % d})")
    print(f"total = {total} {'==' if total == num else '!='} {num} -> {total == num}")
    return total == num

trace_perfect(28)
print()
trace_perfect(36)

num = 28, isqrt = 5
khởi tạo total = 1 (ước 1)
  d=2: chia hết, +2 +14                                   total=17
  d=3: không chia hết (dư 1)
  d=4: chia hết, +4 +7                                    total=28
  d=5: không chia hết (dư 3)
total = 28 == 28 -> True

num = 36, isqrt = 6
khởi tạo total = 1 (ước 1)
  d=2: chia hết, +2 +18                                   total=21
  d=3: chia hết, +3 +12                                   total=36
  d=4: chia hết, +4 +9                                    total=49
  d=5: không chia hết (dư 1)
  d=6: chia hết, +6 (số chính phương, không cộng trùng)   total=55
total = 55 != 36 -> False


False

Với `36`: tổng ước thực sự `1+2+3+4+6+9+12+18 = 55 ≠ 36`. Ở `d = 6`, cặp `(6, 6)` chỉ cộng **một** lần.

## 7. Cách 4 — Định lý Euclid–Euler (kiến thức toán thú vị)

Mọi số hoàn hảo **chẵn** có dạng `2^(p-1) · (2^p − 1)` với `2^p − 1` là **số nguyên tố Mersenne**. Chưa ai tìm thấy số hoàn hảo **lẻ** (đã kiểm chứng không có số nào dưới `10^1500`).

Với `num ≤ 10^8` chỉ có 5 số: `6, 28, 496, 8128, 33550336`. Ta **sinh** chúng bằng công thức để tự kiểm chứng thay vì học thuộc:

In [10]:
def is_prime(n):
    if n < 2:
        return False
    for d in range(2, math.isqrt(n) + 1):
        if n % d == 0:
            return False
    return True

perfects = []
for p in range(2, 32):
    m = 2**p - 1
    if is_prime(m):
        x = 2**(p - 1) * m
        if x <= 10**8:
            perfects.append(x)
            print(f"p={p:2d}: 2^p-1={m:>9} nguyên tố -> {x}")
print("Số hoàn hảo <= 10^8:", perfects)
assert all(is_perfect_sqrt(x) for x in perfects)

p= 2: 2^p-1=        3 nguyên tố -> 6
p= 3: 2^p-1=        7 nguyên tố -> 28
p= 5: 2^p-1=       31 nguyên tố -> 496
p= 7: 2^p-1=      127 nguyên tố -> 8128
p=13: 2^p-1=     8191 nguyên tố -> 33550336
Số hoàn hảo <= 10^8: [6, 28, 496, 8128, 33550336]


In [11]:
PERFECTS = {6, 28, 496, 8128, 33550336}

def is_perfect_lookup(num):
    return num in PERFECTS

print(is_perfect_lookup(28), is_perfect_lookup(7))

True False


Cách tra bảng O(1) đúng với constraints, nhưng **không dạy được kỹ năng**; trong phỏng vấn hãy trình bày cách √n và nhắc Euclid–Euler như kiến thức bổ sung.

## 8. Code nộp LeetCode

### Bản ghép cặp ước (khuyên dùng)

In [12]:
class Solution:
    def checkPerfectNumber(self, num: int) -> bool:
        if num <= 1:
            return False
        total = 1
        d = 2
        while d * d <= num:
            if num % d == 0:
                total += d
                if d != num // d:
                    total += num // d
            d += 1
        return total == num

s = Solution()
print(s.checkPerfectNumber(28), s.checkPerfectNumber(7))   # True False

True False


Ở bản nộp, ta dùng `while d * d <= num` — cách viết phổ biến tương đương `d <= isqrt(num)` mà không cần import.

### Bản tra bảng (Euclid–Euler)

In [13]:
class SolutionLookup:
    def checkPerfectNumber(self, num: int) -> bool:
        return num in (6, 28, 496, 8128, 33550336)

print(SolutionLookup().checkPerfectNumber(496))

True


## 9. Vì sao đúng?

- Mọi ước `d` của `num` ghép với `num // d`; trong mỗi cặp, số nhỏ hơn `≤ √num`. Duyệt `d` từ `1..⌊√num⌋` gặp **mọi cặp đúng một lần**.
- Cặp `(1, num)`: ta lấy `1`, bỏ `num` (không tính chính nó). Đó là lý do khởi tạo `total = 1` và bắt đầu từ `d = 2`.
- Cặp `(d, d)` của số chính phương: chỉ cộng một lần nhờ `if d != num // d`.
- `num = 1`: cặp `(1, 1)` là chính nó, ước thực sự rỗng → tổng 0 ≠ 1 → `False`; ta trả sớm để không bị `total = 1` đánh lừa.

**Invariant:** sau khi xét xong `d`, `total` = tổng mọi ước thực sự của `num` nằm trong các cặp có số nhỏ `≤ d`.

## 10. Độ phức tạp

| Cách | Time | Space | Với num = 10^8 |
|---|---:|---:|---|
| Duyệt 1..num-1 | `O(n)` | `O(1)` | ~10^8 phép chia — quá chậm |
| Duyệt 1..num/2 | `O(n)` | `O(1)` | ~5·10^7 — vẫn chậm |
| Ghép cặp tới √num | `O(√n)` | `O(1)` | ~10^4 — rất nhanh |
| Tra bảng Euclid–Euler | `O(1)` | `O(1)` | tức thì, nhưng dựa vào kiến thức ngoài |

## 11. Bộ test

In [14]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

s = Solution()
check("đề 1: 28", s.checkPerfectNumber(28), True)
check("đề 2: 7", s.checkPerfectNumber(7), False)
check("num = 1", s.checkPerfectNumber(1), False)
check("6", s.checkPerfectNumber(6), True)
check("496", s.checkPerfectNumber(496), True)
check("8128", s.checkPerfectNumber(8128), True)
check("33550336", s.checkPerfectNumber(33550336), True)
check("số chính phương 36", s.checkPerfectNumber(36), False)
check("số nguyên tố 2", s.checkPerfectNumber(2), False)
check("10^8", s.checkPerfectNumber(10**8), False)
check("số dư thừa 12 (tổng ước 16)", s.checkPerfectNumber(12), False)

PASS đề 1: 28: True
PASS đề 2: 7: False
PASS num = 1: False
PASS 6: True
PASS 496: True
PASS 8128: True
PASS 33550336: True
PASS số chính phương 36: False
PASS số nguyên tố 2: False
PASS 10^8: False
PASS số dư thừa 12 (tổng ước 16): False


In [15]:
# Đối chiếu brute force với các cách nhanh trên toàn đoạn [1, 20000]
for n in range(1, 20001):
    expected = is_perfect_bruteforce(n) if n < 3000 else (n in PERFECTS)
    assert is_perfect_sqrt(n) == expected == is_perfect_lookup(n) == Solution().checkPerfectNumber(n), n
for n in range(1, 3000):
    assert is_perfect_half(n) == is_perfect_bruteforce(n), n
print("[1, 20000]: các cách khớp nhau — PASS")

[1, 20000]: các cách khớp nhau — PASS


## 12. Lỗi người mới hay gặp

### Lỗi 1: quên `num = 1`
Khởi tạo `total = 1` rồi trả `total == num` → `1` được coi là hoàn hảo (sai).

### Lỗi 2: đếm trùng ước của số chính phương
Cộng cả `d` và `num // d` khi chúng bằng nhau.

### Lỗi 3: cộng luôn cả `num`
Duyệt `d` từ 1 và cộng cả `num // 1 = num` → tổng luôn > num.

### Lỗi 4: dùng `num / d`
Ra số thực (`7.0`), tổng thành `float` — so sánh vẫn có thể đúng nhưng dễ gây sai số và khó đọc.

In [16]:
def buggy_no_one_check(num):
    total = 1
    d = 2
    while d * d <= num:
        if num % d == 0:
            total += d + (num // d if d != num // d else 0)
        d += 1
    return total == num

def buggy_double_count(num):
    if num <= 1:
        return False
    total = 1
    d = 2
    while d * d <= num:
        if num % d == 0:
            total += d + num // d          # luôn cộng cả hai
        d += 1
    return total, total == num

def buggy_include_self(num):
    total = 0
    d = 1
    while d * d <= num:
        if num % d == 0:
            total += d + (num // d if d != num // d else 0)
        d += 1
    return total, total == num

print("quên num=1     :", buggy_no_one_check(1), "| đúng:", Solution().checkPerfectNumber(1))
print("đếm trùng (36) : tổng =", buggy_double_count(36)[0], "| đúng là 55")
print("cộng cả num(28):", buggy_include_self(28), "| tổng đúng là 28")

quên num=1     : True | đúng: False
đếm trùng (36) : tổng = 61 | đúng là 55
cộng cả num(28): (56, False) | tổng đúng là 28


## 13. Mẹo nhận dạng pattern

- "Ước số", "tổng/đếm ước", "số nguyên tố", "phân tích thừa số" → **duyệt tới `√n`** và ghép cặp `(d, n // d)`.
- Luôn kiểm tra: `n = 1`, số **chính phương** (`d == n // d`), số **nguyên tố** (chỉ có cặp `(1, n)`).
- Constraints `≤ 10^8`/`10^9` và chỉ một truy vấn → `O(√n)` là đủ.
- Nhiều truy vấn trên cả đoạn `[1, N]` → nghĩ tới **sàng** (sieve) cộng ước cho mọi số trong `O(N log N)`.

## 14. Biến thể

### 14.1. Phân loại: hoàn hảo / dư thừa / thiếu hụt
- **Abundant** (dư thừa): tổng ước thực sự `> n` (ví dụ 12).
- **Deficient** (thiếu hụt): `< n` (ví dụ 7).

In [17]:
def proper_divisor_sum(n):
    if n <= 1:
        return 0
    total, d = 1, 2
    while d * d <= n:
        if n % d == 0:
            total += d + (n // d if d != n // d else 0)
        d += 1
    return total

def classify(n):
    s = proper_divisor_sum(n)
    return "perfect" if s == n else ("abundant" if s > n else "deficient")

print([(n, classify(n)) for n in [6, 7, 12, 28, 1]])

[(6, 'perfect'), (7, 'deficient'), (12, 'abundant'), (28, 'perfect'), (1, 'deficient')]


Kiểm tra **tính chất** của `proper_divisor_sum` (không cần biết đáp án trước): với số nguyên tố `p`, tổng ước thực sự luôn là `1`; với `p²`, tổng là `1 + p`.

In [18]:
import random
random.seed(507)
primes = [p for p in range(2, 5000) if is_prime(p)]
for p in random.sample(primes, 50):
    assert proper_divisor_sum(p) == 1
    assert proper_divisor_sum(p * p) == 1 + p
    assert Solution().checkPerfectNumber(p) is False
print("50 số nguyên tố ngẫu nhiên: tính chất tổng ước đúng — PASS")

50 số nguyên tố ngẫu nhiên: tính chất tổng ước đúng — PASS


### 14.2. Sàng tổng ước cho mọi số ≤ N
Với mỗi `d`, cộng `d` vào mọi bội `2d, 3d, ...`. Tổng số thao tác `N/1 + N/2 + ... ≈ N ln N`.

In [19]:
def divisor_sum_sieve(N):
    s = [0] * (N + 1)
    for d in range(1, N // 2 + 1):
        for m in range(2 * d, N + 1, d):
            s[m] += d
    return s

sv = divisor_sum_sieve(10000)
print("số hoàn hảo <= 10000 (sàng):", [n for n in range(2, 10001) if sv[n] == n])

số hoàn hảo <= 10000 (sàng): [6, 28, 496, 8128]


## 15. Bài tập (đáp án ở cuối)

### Bài 1 — Tính tay
Liệt kê các cặp ước của `496` khi duyệt `d` tới `⌊√496⌋ = 22`, và tính tổng ước thực sự.

### Bài 2 — Điền TODO

In [20]:
def exercise_is_perfect(num):
    # TODO 1: num <= 1 -> False
    total = 1
    d = 2
    while d * d <= num:
        # TODO 2: nếu d là ước -> cộng d và (num // d nếu khác d)
        d += 1
    # TODO 3: so sánh
    pass

# assert exercise_is_perfect(28) is True
# assert exercise_is_perfect(36) is False

### Bài 3 — Đếm số ước
Trả về số ước dương của `n` (tính cả 1 và `n`) trong `O(√n)`.

In [21]:
def count_divisors(n):
    # TODO
    pass

# assert count_divisors(36) == 9
# assert count_divisors(13) == 2

### Bài 4 — Kiểm tra số nguyên tố trong O(√n)

In [22]:
def exercise_is_prime(n):
    # TODO
    pass

# assert [x for x in range(20) if exercise_is_prime(x)] == [2, 3, 5, 7, 11, 13, 17, 19]

### Bài 5 — Cặp số thân thiết (amicable numbers)
`a ≠ b` là cặp thân thiết nếu tổng ước thực sự của `a` bằng `b` và ngược lại. Trả về list cặp `(a, b)` với `a < b ≤ limit`.

In [23]:
def amicable_pairs(limit):
    # TODO
    pass

# assert amicable_pairs(1500) == [(220, 284), (1184, 1210)]

### Bài 6 — Mini challenge: số hoàn hảo trong đoạn bằng sàng
Trả về các số hoàn hảo `≤ N` bằng sàng tổng ước (không dùng bảng có sẵn).

In [24]:
def perfect_numbers_upto(N):
    # TODO
    pass

# assert perfect_numbers_upto(10000) == [6, 28, 496, 8128]

## 16. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | Thử `d = 2, 4, 8, 16` | `496 = 16 · 31` | dùng `trace_perfect(496)` để kiểm |
| 2 | Xem mục 5 | `if num % d == 0:` | `total += d + (num // d if d != num // d else 0)` |
| 3 | Mỗi cặp đóng góp 2 ước | trừ khi `d == n // d` | `cnt += 1 if d == n // d else 2` |
| 4 | Có ước nào trong `2..√n`? | `n < 2` → False | `return all(n % d for d in range(2, isqrt(n) + 1))` |
| 5 | Dùng `proper_divisor_sum` | `b = s(a)`; kiểm `b > a and s(b) == a` | điều kiện `b > a` tránh trùng cặp và loại số hoàn hảo |
| 6 | Dùng sàng mục 14.2 | so `s[n] == n` | bỏ qua `n = 1` |

## 17. Đáp án tham khảo

**Bài 1:** các cặp `(1, 496), (2, 248), (4, 124), (8, 62), (16, 31)` → tổng ước thực sự `1+2+4+8+16+31+62+124+248 = 496` → hoàn hảo.

In [25]:
trace_perfect(496)

num = 496, isqrt = 22
khởi tạo total = 1 (ước 1)
  d=2: chia hết, +2 +248                                  total=251
  d=3: không chia hết (dư 1)
  d=4: chia hết, +4 +124                                  total=379
  d=5: không chia hết (dư 1)
  d=6: không chia hết (dư 4)
  d=7: không chia hết (dư 6)
  d=8: chia hết, +8 +62                                   total=449
  d=9: không chia hết (dư 1)
  d=10: không chia hết (dư 6)
  d=11: không chia hết (dư 1)
  d=12: không chia hết (dư 4)
  d=13: không chia hết (dư 2)
  d=14: không chia hết (dư 6)
  d=15: không chia hết (dư 1)
  d=16: chia hết, +16 +31                                  total=496
  d=17: không chia hết (dư 3)
  d=18: không chia hết (dư 10)
  d=19: không chia hết (dư 2)
  d=20: không chia hết (dư 16)
  d=21: không chia hết (dư 13)
  d=22: không chia hết (dư 12)
total = 496 == 496 -> True


True

In [26]:
def answer_exercise_is_perfect(num):
    if num <= 1:
        return False
    total, d = 1, 2
    while d * d <= num:
        if num % d == 0:
            total += d + (num // d if d != num // d else 0)
        d += 1
    return total == num

def answer_count_divisors(n):
    cnt, d = 0, 1
    while d * d <= n:
        if n % d == 0:
            cnt += 1 if d == n // d else 2
        d += 1
    return cnt

def answer_is_prime(n):
    if n < 2:
        return False
    return all(n % d for d in range(2, math.isqrt(n) + 1))

def answer_amicable_pairs(limit):
    res = []
    for a in range(2, limit + 1):
        b2 = proper_divisor_sum(a)
        if a < b2 <= limit and proper_divisor_sum(b2) == a:
            res.append((a, b2))
    return res

def answer_perfect_numbers_upto(N):
    s = divisor_sum_sieve(N)
    return [n for n in range(2, N + 1) if s[n] == n]

assert answer_exercise_is_perfect(28) is True and answer_exercise_is_perfect(36) is False
assert answer_count_divisors(36) == 9 and answer_count_divisors(13) == 2
assert [x for x in range(20) if answer_is_prime(x)] == [2, 3, 5, 7, 11, 13, 17, 19]
assert answer_amicable_pairs(1500) == [(220, 284), (1184, 1210)]
assert answer_perfect_numbers_upto(10000) == [6, 28, 496, 8128]
print("Đáp án bài 2–6: PASS toàn bộ test!")

Đáp án bài 2–6: PASS toàn bộ test!


## 18. Template nhớ nhanh + checklist submit

```python
if num <= 1:
    return False
total, d = 1, 2                      # ước 1 tính sẵn, bỏ chính num
while d * d <= num:                  # chỉ tới √num
    if num % d == 0:
        total += d
        if d != num // d:            # số chính phương: không cộng trùng
            total += num // d
    d += 1
return total == num
```

Câu nhớ: **"Ước đi theo cặp — duyệt nửa nhỏ, cộng cả nửa lớn."**

### Checklist
- [ ] `num = 1` → `False`.
- [ ] Không cộng chính `num`.
- [ ] Số chính phương không đếm trùng.
- [ ] Dùng `//` cho ước đối xứng.
- [ ] Time `O(√num)`, Space `O(1)`.